# Final Project Task 1 — Data Exploration
## 보유 재료 기반 레시피 추천 서비스 (가칭: 냉장고 털이)

- **선택 데이터**: 식품의약품안전처 「조리식품의 레시피 DB」 (OpenAPI 서비스 ID: `COOKRCP01`)
- **비교 후보**: Food.com Recipes and Interactions (Kaggle)
- **팀원**: 최승수, 조은우, 장윤우, 조민혁

이 노트북은 데이터를 API로 받아 저장한 뒤, 보고서에 들어가는 모든 통계·표·그래프를 다시 만들어 냅니다.
위에서부터 순서대로 **Run All** 하면 됩니다. 실행 방법은 `README.md`를 보세요.

## 0. 설정

In [ ]:
import os, re, json, time
from datetime import datetime
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import requests

# 한글 폰트 (Windows: Malgun Gothic / macOS: AppleGothic / Linux: Nanum 또는 Noto CJK)
plt.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

DATA_DIR = "data"
FIG_DIR = "figures"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

RAW_PATH = os.path.join(DATA_DIR, "recipes_raw.csv")
INFO_PATH = os.path.join(DATA_DIR, "download_info.json")

# API 키는 코드에 직접 쓰지 말고 환경변수로 넣으세요 (README 참고)
API_KEY = os.environ.get("FOOD_API_KEY", "")

## 1. 데이터 수집 (Step 2)
식품안전나라 OpenAPI는 한 번에 최대 1,000건까지 줍니다. 전체 건수(`total_count`)를 먼저 확인하고 1,000건씩 나눠 받습니다.
이미 `data/recipes_raw.csv`가 있으면 다시 받지 않습니다.

In [ ]:
BASE_URL = "http://openapi.foodsafetykorea.go.kr/api/{key}/COOKRCP01/json/{start}/{end}"

def fetch_page(start, end):
    url = BASE_URL.format(key=API_KEY, start=start, end=end)
    res = requests.get(url, timeout=30)
    res.raise_for_status()
    body = res.json()["COOKRCP01"]
    code_ = body.get("RESULT", {}).get("CODE", "")
    if code_ not in ("INFO-000", ""):
        raise RuntimeError(f"API 오류: {body.get('RESULT')}")
    return int(body["total_count"]), body.get("row", [])

def download_all(step=1000):
    total, _ = fetch_page(1, 1)
    rows = []
    for start in range(1, total + 1, step):
        end = min(start + step - 1, total)
        _, page = fetch_page(start, end)
        rows.extend(page)
        print(f"{start}~{end} 수신, 누적 {len(rows)}건")
        time.sleep(0.5)
    return total, rows

if not os.path.exists(RAW_PATH):
    if not API_KEY:
        raise SystemExit("FOOD_API_KEY 환경변수를 설정하세요. (README 참고)")
    total, rows = download_all()
    pd.DataFrame(rows).to_csv(RAW_PATH, index=False, encoding="utf-8-sig")
    with open(INFO_PATH, "w", encoding="utf-8") as f:
        json.dump({"downloaded_at": datetime.now().isoformat(timespec="seconds"),
                   "api_total_count": total, "rows_received": len(rows),
                   "source": "식품의약품안전처 조리식품의 레시피 DB (COOKRCP01)"}, f, ensure_ascii=False, indent=2)
    print(f"저장 완료: {RAW_PATH}")
else:
    print(f"기존 파일 사용: {RAW_PATH}")

if os.path.exists(INFO_PATH):
    print(open(INFO_PATH, encoding="utf-8").read())

## 2. 불러오기와 기본 점검 (Step 2)
원본을 그대로 보기 위해 모든 컬럼을 문자열로 읽고, 빈 문자열도 그대로 둡니다(이 API는 결측을 `null`이 아니라 빈 문자열로 줍니다).

In [ ]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")

COLS = {
    "RCP_SEQ": "일련번호", "RCP_NM": "메뉴명", "RCP_WAY2": "조리방법", "RCP_PAT2": "요리종류",
    "INFO_WGT": "중량(1인분)", "INFO_ENG": "열량", "INFO_CAR": "탄수화물", "INFO_PRO": "단백질",
    "INFO_FAT": "지방", "INFO_NA": "나트륨", "HASH_TAG": "해쉬태그", "RCP_PARTS_DTLS": "재료정보",
    "RCP_NA_TIP": "저감조리법TIP",
}
manual_cols = sorted([c for c in raw.columns if re.fullmatch(r"MANUAL\d{2}", c)])

print("행, 열 (원본 전체):", raw.shape)
print("API 응답 컬럼 목록:", list(raw.columns))

df = raw[[c for c in COLS if c in raw.columns]].rename(columns=COLS).copy()
df["조리단계수"] = (raw[manual_cols].apply(lambda s: s.str.strip() != "")).sum(axis=1) if manual_cols else 0
print("\n분석용 데이터 행, 열:", df.shape)
df.dtypes

In [ ]:
# 대표 레코드
df.head(5)

### 2-1. 결측값 (빈 문자열 포함)

In [ ]:
blank = df.apply(lambda s: s.astype(str).str.strip().eq("") | s.isna())
missing = pd.DataFrame({"결측 수": blank.sum(), "결측 비율(%)": (blank.mean() * 100).round(2)})
missing.sort_values("결측 수", ascending=False)

### 2-2. 중복

In [ ]:
dup_full = raw.duplicated().sum()
dup_seq = df["일련번호"].duplicated().sum()
dup_name = df["메뉴명"].str.strip().duplicated(keep=False)
print("완전히 같은 행:", dup_full)
print("일련번호 중복:", dup_seq)
print("메뉴명이 겹치는 행:", dup_name.sum())
df.loc[dup_name, ["일련번호", "메뉴명", "요리종류", "열량"]].sort_values("메뉴명").head(20)

### 2-3. 영양 컬럼: 문자열 → 숫자 변환과 기초 통계

In [ ]:
NUM_COLS = ["중량(1인분)", "열량", "탄수화물", "단백질", "지방", "나트륨"]
num_report = []
for c in NUM_COLS:
    if c not in df.columns:
        continue
    s = df[c].astype(str).str.strip()
    conv = pd.to_numeric(s, errors="coerce")
    num_report.append({"컬럼": c, "빈 값": int(s.eq("").sum()),
                       "숫자 변환 실패(빈 값 제외)": int(conv.isna().sum() - s.eq("").sum()),
                       "0인 값": int((conv == 0).sum())})
    df[c + "_num"] = conv
display(pd.DataFrame(num_report)) if "display" in globals() else print(pd.DataFrame(num_report))

# 변환에 실패한 원본 값 예시 (단위가 섞여 있는지 확인)
for c in NUM_COLS:
    if c in df.columns:
        bad = df.loc[df[c + "_num"].isna() & df[c].str.strip().ne(""), c].unique()[:10]
        if len(bad):
            print(c, "변환 실패 예:", list(bad))

df[[c + "_num" for c in NUM_COLS if c in df.columns]].describe().round(1)

### 2-4. 범주형 컬럼 분포

In [ ]:
print(df["요리종류"].value_counts(), "\n")
print(df["조리방법"].value_counts())

## 3. 핵심 데이터 품질 문제: `재료정보` (Step 2)
서비스의 핵심 기능(가진 재료와 레시피 재료 비교)은 이 컬럼에 달려 있습니다.
그런데 재료가 **하나의 긴 문자열**로 들어 있고, 양·단위·소제목(예: "양념", "고명")이 섞여 있습니다.

In [ ]:
for i, row in df.sample(5, random_state=1).iterrows():
    print(f"[{row['메뉴명']}]")
    print(repr(row["재료정보"][:300]))
    print()

### 3-1. 재료 문자열을 재료 이름 목록으로 나누기
1. 줄바꿈과 쉼표로 나눔 → 2. `:` 앞의 소제목 제거 → 3. 괄호 안 내용 제거
→ 4. 숫자·분수·단위가 시작되는 지점에서 자름 → 5. "약간", "적당량" 같은 말과 소제목 단어 제거 → 6. 메뉴명과 같은 줄(제목 줄) 제거

In [ ]:
SECTION_WORDS = {"재료", "주재료", "부재료", "양념", "양념장", "소스", "고명", "드레싱", "육수", "반죽",
                 "속재료", "곁들임", "토핑", "장식", "기타", "필수재료", "선택재료"}
AMOUNT_WORDS = r"(약간|적당량|적당히|조금|소량|적량|기호에\s*따라|취향껏|선택)"

def split_ingredients(text, menu_name=""):
    if not isinstance(text, str) or not text.strip():
        return []
    parts = re.split(r"[\n,]", text)
    out = []
    for p in parts:
        p = p.strip()
        p = re.sub(r"[●•▶■□◆◇\-\*\[\]]", " ", p)
        if ":" in p:
            p = p.split(":", 1)[1]
        p = re.sub(r"\(.*?\)", " ", p)
        p = re.split(r"[\d½⅓¼¾]", p, maxsplit=1)[0]
        p = re.sub(AMOUNT_WORDS, " ", p)
        p = re.sub(r"\s+", " ", p).strip()
        if not p or p in SECTION_WORDS:
            continue
        if menu_name and p.replace(" ", "") == menu_name.replace(" ", ""):
            continue   # 제목 줄
        if len(p) > 15:
            continue   # 문장(설명) 조각
        out.append(p)
    return out

df["재료목록_원본"] = [split_ingredients(t, n) for t, n in zip(df["재료정보"], df["메뉴명"])]
df["재료수"] = df["재료목록_원본"].str.len()
df[["메뉴명", "재료목록_원본", "재료수"]].head(10)

### 3-2. 같은 재료가 여러 이름으로 적힌 문제 (표기 불일치)
손질 상태("다진", "채 썬")나 동의어("계란"/"달걀") 때문에 같은 재료가 다른 이름으로 집계됩니다.
이대로 매칭하면 사용자가 "마늘"을 입력해도 "다진 마늘"이 들어간 레시피가 빠집니다.

In [ ]:
PREP_WORDS = r"^(다진|채썬|채 썬|송송 썬|어슷 썬|삶은|데친|볶은|구운|불린|말린|냉동|손질한|으깬)\s*"   # '간', '생', '깐'은 간장·생강·깐풍기를 망가뜨려서 제외
SYNONYMS = {
    "계란": "달걀", "달걀노른자": "달걀", "달걀흰자": "달걀", "계란노른자": "달걀", "계란흰자": "달걀",
    "마늘즙": "마늘", "깐마늘": "마늘", "간마늘": "마늘",
    "진간장": "간장", "국간장": "간장", "양조간장": "간장",
    "백설탕": "설탕", "황설탕": "설탕",
    "후춧가루": "후추", "통후추": "후추",
    "참깨": "깨", "통깨": "깨",
    "식용유": "기름", "포도씨유": "기름", "카놀라유": "기름",
}   # Task 2에서 팀이 계속 보완할 동의어 사전

def normalize(name):
    n = re.sub(PREP_WORDS, "", name).replace(" ", "")
    return SYNONYMS.get(n, n)

df["재료목록"] = df["재료목록_원본"].apply(lambda xs: sorted({normalize(x) for x in xs}))

all_raw = [x.replace(" ", "") for xs in df["재료목록_원본"] for x in xs]
all_norm = [x for xs in df["재료목록"] for x in xs]
print("재료 정보가 비어 있거나 나눌 수 없는 레시피:", int((df["재료수"] == 0).sum()))
print("원본 재료 표기 종류 수:", len(set(all_raw)))
print("정규화 후 재료 종류 수:", len(set(all_norm)))
print("한 번만 등장하는 재료(정규화 후):", sum(1 for v in Counter(all_norm).values() if v == 1))

# 예시: '마늘'과 '달걀'이 원본에서 몇 가지 이름으로 적혀 있었는지
for key in ["마늘", "달걀", "계란", "간장", "파"]:
    variants = Counter(x for x in all_raw if key in x)
    print(f"\n'{key}' 포함 표기 {len(variants)}종:", variants.most_common(8))

## 4. 탐색적 분석 (Step 3)
### 4-1. 요약 표: 요리종류별 레시피 수와 영양·재료 수
※ 레시피 수는 이 DB에 수록된 건수일 뿐, 요리의 인기도가 아닙니다.

In [ ]:
summary = (df.groupby("요리종류")
             .agg(레시피수=("메뉴명", "count"),
                  평균열량_kcal=("열량_num", "mean"),
                  중앙값나트륨_mg=("나트륨_num", "median"),
                  평균단백질_g=("단백질_num", "mean"),
                  평균재료수=("재료수", "mean"))
             .round(1)
             .sort_values("레시피수", ascending=False))
summary.to_csv(os.path.join(FIG_DIR, "summary_by_category.csv"), encoding="utf-8-sig")
summary

### 4-2. 시각화: 가장 많은 레시피에 들어가는 재료 TOP 20

In [ ]:
n_recipes = int((df["재료수"] > 0).sum())
ing_count = Counter(x for xs in df["재료목록"] for x in xs)   # 레시피당 1번만 셈
top = pd.Series(ing_count).sort_values(ascending=False).head(20)
top_pct = (top / n_recipes * 100).round(1)

SEASONINGS = {"소금", "설탕", "간장", "후추", "참기름", "기름", "깨", "물", "고춧가루", "고추장",
              "된장", "식초", "올리고당", "물엿", "맛술", "들기름", "꿀", "청주", "굴소스", "버터", "전분"}
colors = ["#9AA5B1" if k in SEASONINGS else "#2F6FB0" for k in top_pct.index]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_pct.index[::-1], top_pct.values[::-1], color=colors[::-1])
ax.set_xlabel("이 재료가 들어간 레시피 비율 (%)")
ax.set_title(f"레시피에 가장 자주 쓰이는 재료 TOP 20 (n={n_recipes})\n회색 = 기본 양념류, 파랑 = 주재료")
for y, v in enumerate(top_pct.values[::-1]):
    ax.text(v + 0.3, y, f"{v}%", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "top20_ingredients.png"), dpi=200)
plt.show()

pd.DataFrame({"레시피 수": top, "비율(%)": top_pct, "양념류": [k in SEASONINGS for k in top.index]})

### 4-3. 보조 분석: 레시피당 재료 수 분포

In [ ]:
valid = df[df["재료수"] > 0]
main_cnt = valid["재료목록"].apply(lambda xs: len([x for x in xs if x not in SEASONINGS]))
print("레시피당 재료 수(정규화 후):")
print(valid["재료목록"].str.len().describe().round(1))
print("\n레시피당 주재료 수(양념 제외):")
print(main_cnt.describe().round(1))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist([valid["재료목록"].str.len(), main_cnt], bins=range(0, 31), label=["전체 재료", "양념 제외 주재료"], alpha=0.7)
ax.set_xlabel("레시피당 재료 수"); ax.set_ylabel("레시피 수"); ax.legend()
ax.set_title("레시피당 재료 수 분포")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "ingredients_per_recipe.png"), dpi=200); plt.show()

### 4-4. 관찰을 뒷받침하는 숫자
아래 출력값을 근거로 보고서의 **관찰 2개**를 작성합니다. (예: 상위 재료 중 양념류 비율, 주재료 수의 중앙값)

In [ ]:
share_seasoning_top20 = sum(k in SEASONINGS for k in top.index)
print(f"TOP 20 재료 중 양념류: {share_seasoning_top20}개")
print(f"양념 제외 주재료 수 중앙값: {main_cnt.median():.0f}개, 75% 분위: {main_cnt.quantile(0.75):.0f}개")
print("요리종류별 열량 차이 (최대-최소 평균):",
      round(summary['평균열량_kcal'].max() - summary['평균열량_kcal'].min(), 1), "kcal")
print("요리종류별 나트륨 중앙값 최고:", summary["중앙값나트륨_mg"].idxmax(), summary["중앙값나트륨_mg"].max(), "mg")

## 5. 서비스 핵심 기능 실현 가능성 확인 (Step 4~5)
**입력**: 사용자가 가진 재료 → **처리**: 레시피별 주재료(양념 제외) 일치율 계산 → **결과**: 일치율 높은 순으로 레시피, 부족한 재료, 열량.
Task 1에서는 웹 구현이 아니라 데이터로 이 흐름이 가능한지만 확인합니다.

In [ ]:
def recommend(user_ingredients, top_k=10, assume_seasonings=True):
    have = {normalize(x) for x in user_ingredients}
    results = []
    for _, r in df[df["재료수"] > 0].iterrows():
        need = [x for x in r["재료목록"] if not (assume_seasonings and x in SEASONINGS)]
        if not need:
            continue
        matched = [x for x in need if x in have]
        results.append({"메뉴명": r["메뉴명"], "요리종류": r["요리종류"],
                        "일치율(%)": round(len(matched) / len(need) * 100, 1),
                        "가진 재료": ", ".join(matched),
                        "부족한 재료": ", ".join(x for x in need if x not in have),
                        "열량": r["열량_num"]})
    out = pd.DataFrame(results).drop_duplicates("메뉴명")   # 같은 메뉴가 여러 번 수록된 경우 한 번만 표시
    return out.sort_values(["일치율(%)", "열량"], ascending=[False, True]).head(top_k)

example_input = ["달걀", "양파", "대파", "두부"]
print("입력 예시:", example_input)
recommend(example_input)

In [ ]:
# 입력이 바뀌면 결과가 바뀌는지 확인 (서비스 조건)
recommend(["닭가슴살", "양배추", "당근"])

## 6. 보고서용 핵심 숫자 모음

In [ ]:
report_numbers = {
    "원본 행 수": raw.shape[0], "원본 열 수": raw.shape[1],
    "분석용 컬럼 수": len([c for c in COLS.values() if c in df.columns]),
    "재료정보 빈 레시피 수": int((df["재료수"] == 0).sum()),
    "메뉴명 중복 행 수": int(dup_name.sum()),
    "원본 재료 표기 종류": len(set(all_raw)),
    "정규화 후 재료 종류": len(set(all_norm)),
    "TOP20 중 양념류 개수": share_seasoning_top20,
    "주재료 수 중앙값": float(main_cnt.median()),
}
if os.path.exists(INFO_PATH):
    report_numbers.update(json.load(open(INFO_PATH, encoding="utf-8")))
for k, v in report_numbers.items():
    print(f"{k}: {v}")

## (부록) 비교 후보: Food.com Recipes and Interactions
Kaggle에서 `RAW_recipes.csv`를 받아 `data/` 폴더에 넣으면 아래 셀이 기본 점검을 합니다. 파일이 없으면 건너뜁니다.
- https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions

In [ ]:
FOODCOM_PATH = os.path.join(DATA_DIR, "RAW_recipes.csv")
if os.path.exists(FOODCOM_PATH):
    fc = pd.read_csv(FOODCOM_PATH)
    print("행, 열:", fc.shape)
    print(fc.dtypes)
    print(fc.isna().sum())
    print(fc[["name", "minutes", "n_ingredients"]].describe())
    display(fc.head(3)) if "display" in globals() else print(fc.head(3))
else:
    print("Food.com 파일 없음 → 건너뜀 (보고서의 후보 비교는 공개된 데이터셋 설명을 근거로 작성)")